
# <font color="blue">Segmentación de Instancias con YOLO11</h1>
### Entrenamiento con dataset personalizado desde Roboflow
# <font color="red">Alfredo Diaz Claro</h2> </font>
---

> **Adaptado por:** Alfredo Diaz  
> **Basado en:** [Roboflow Notebooks](https://github.com/roboflow/notebooks)  
> **Modelo:** YOLO11 — Ultralytics  
> **Tarea:** Instance Segmentation (Segmentación de Instancias)

---

## ¿Qué aprenderás en este notebook?

Este notebook te guía paso a paso para:

1. Entender qué es la **segmentación de instancias** y por qué es importante
2. Preparar y descargar un **dataset anotado desde Roboflow**
3. **Entrenar YOLO11** para segmentación con tu propio dataset
4. Aplicar **Early Stopping** para evitar sobreajuste y reducir tiempo de entrenamiento
5. **Validar e inferir** con el modelo entrenado
6. **Desplegar** el modelo en Roboflow o usarlo localmente con OpenCV

---

##  ¿Qué es la Segmentación de Instancias?

Existen diferentes niveles de comprensión visual en visión por computadora:

| Técnica | ¿Qué hace? | Ejemplo |
|---|---|---|
| **Clasificación** | Dice *qué* hay en la imagen | "Hay un perro" |
| **Detección (Bounding Box)** | Dice *dónde* está con un rectángulo | Caja alrededor del perro |
| **Segmentación Semántica** | Colorea píxeles por categoría (todos los perros = mismo color) | Máscara de clase sin distinguir cuál objeto |
| **Segmentación de Instancias** | Colorea cada objeto individualmente | Perro 1 en azul, Perro 2 en rojo |

**La segmentación de instancias** es la técnica más completa: no solo detecta el objeto sino que genera una **máscara poligonal** que sigue exactamente su contorno. Esto es fundamental en:

-  Control de calidad industrial (detectar defectos exactos en piezas)
-  Vehículos autónomos (separar peatones, autos, señales con precisión)
-  Análisis médico (segmentar tumores, células)
-  Agricultura de precisión (contar frutas, detectar plagas)
-  Detección de daños en vehículos

---

##  ¿Cómo se anota un dataset para segmentación en Roboflow?

**Roboflow** es una plataforma que permite preparar datasets de visión por computadora de forma profesional:
- Subir y organizar imágenes
- Anotar objetos con **polígonos** (no rectángulos)
- Aplicar aumentaciones de datos automáticas
- Exportar en el formato exacto que necesita YOLO

### Pasos para anotar en Roboflow:

1. **Crear un proyecto** → Tipo: *Instance Segmentation*
2. **Subir imágenes** → Acepta imágenes, videos o URLs
3. **Anotar con polígonos** → Usar *Smart Polygon* (IA integrada) o dibujar el contorno manualmente
4. **Generar versión** → Aplicar aumentaciones: flip, rotación, brillo, recorte, etc.
5. **Exportar** → Seleccionar formato `YOLOv11` → Obtener código de descarga con API Key

### ¿Por qué usar polígonos y no rectángulos?

Los rectángulos (bounding boxes) incluyen píxeles del fondo que no pertenecen al objeto. Los **polígonos** siguen exactamente el contorno, lo que permite:
- Mucho mayor precisión en la segmentación
- Mejores métricas (mAP de máscara más alto)
- Aplicaciones donde el contorno exacto importa: robótica, manufactura, cirugía

---

## ⚙️ 1. Configuración del Entorno

### 1.1 Verificar GPU disponible

YOLO11 entrena mucho más rápido en GPU. En Google Colab:
- Ve a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución`
- Selecciona `GPU (T4)` y guarda

El siguiente comando verifica que la GPU esté activa y muestra su información:

In [ ]:
!nvidia-smi

### 1.2 Definir directorio HOME

Creamos una variable `HOME` con la ruta del directorio actual para organizar todos los archivos del proyecto (datasets, pesos del modelo, resultados) en un mismo lugar.

In [ ]:
import os
HOME = os.getcwd()
print(f" Directorio de trabajo: {HOME}")

### 1.3 Instalar dependencias

Instalamos los tres paquetes principales:
- **`ultralytics`**: Framework oficial de YOLO11. Incluye entrenamiento, inferencia, validación y exportación.
- **`supervision`**: Librería de anotación visual. Facilita dibujar máscaras, polígonos y etiquetas sobre imágenes.
- **`roboflow`**: Cliente para descargar datasets desde Roboflow Universe.

> ⚠️ La opción `sync=False` desactiva la telemetría de Ultralytics para mayor privacidad en el entrenamiento.

In [ ]:
%pip install -q "ultralytics>=8.3.0" supervision roboflow

# Desactivar seguimiento de actividad de Ultralytics
!yolo settings sync=False

import ultralytics
ultralytics.checks()

---

## 🔍 2. Inferencia con Modelo Pre-entrenado en COCO

Antes de entrenar nuestro propio modelo, exploramos YOLO11 con pesos preentrenados en **COCO** (Common Objects in Context: 80 categorías comunes como personas, autos, animales, muebles, etc.).

Esto nos permite:
- Ver cómo se ven las máscaras de segmentación en la práctica
- Entender los datos que devuelve el modelo (coordenadas, confianza, clases, máscaras)
- Comparar resultados antes y después del entrenamiento personalizado

### 2.1 Descargar imágenes de prueba

In [ ]:
# Descargamos imágenes de ejemplo de Roboflow para probar el modelo pre-entrenado
!wget -q https://raw.githubusercontent.com/adiacla/segmentacion/refs/heads/main/imagenes/WhatsApp%20Image%202026-04-06%20at%201.32.35%20PM.jpeg
!wget -q https://raw.githubusercontent.com/adiacla/segmentacion/refs/heads/main/imagenes/WhatsApp%20Image%202026-04-19%20at%206.40.06%20PM.jpeg
print(" Imágenes de prueba descargadas")

In [ ]:
!mv "/content/WhatsApp Image 2026-04-06 at 1.32.35 PM.jpeg" "/content/foto1.jpeg"
!mv "/content/WhatsApp Image 2026-04-19 at 6.40.06 PM.jpeg" "/content/foto2.jpeg"

### 2.2 Inferencia por CLI (línea de comandos)

La CLI de YOLO permite ejecutar inferencia con un solo comando, sin escribir código Python. Es útil para pruebas rápidas.

**Parámetros del comando:**
- `task=segment` → Modo segmentación de instancias
- `model=yolo11m-seg.pt` → Modelo mediano pre-entrenado de YOLO11 para segmentación (se descarga automáticamente)
- `source=` → Ruta de la imagen de entrada
- `save=True` → Guarda la imagen con las máscaras dibujadas

In [ ]:
!yolo task=segment mode=predict model=yolo11m-seg.pt source={HOME}/foto1.jpeg save=True verbose=False

In [ ]:
# Verificar que la imagen anotada fue guardada correctamente
!ls -la {HOME}/runs/segment/predict/

from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/segment/predict/foto1.jpg', width=600)

In [ ]:
!yolo task=segment mode=predict model=yolo11m-seg.pt source={HOME}/foto2.jpeg save=True verbose=False

In [ ]:
# Verificar que la imagen anotada fue guardada correctamente
!ls -la {HOME}/runs/segment/predict/

from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/segment/predict-2/foto2.jpg', width=600)

### 2.3 Inferencia por SDK (Python)

La API de Python ofrece acceso programático completo: podemos leer coordenadas, confianza, clases y máscaras directamente como tensores de PyTorch o NumPy, lo cual es esencial para integrar el modelo en aplicaciones.

In [ ]:
from ultralytics import YOLO
from PIL import Image

# Cargar modelo pre-entrenado YOLO11 para segmentación
model = YOLO('yolo11m-seg.pt')

# Cargar imagen y ejecutar inferencia
image = Image.open(f'{HOME}/foto2.jpeg')
result = model.predict(image, verbose=False)[0]  # [0] porque pasamos 1 imagen

print(" Bounding Boxes [x1, y1, x2, y2]:")
print(result.boxes.xyxy)

print("\n Confianza de cada detección (0 a 1):")
print(result.boxes.conf)

print("\n Índice de clase detectada:")
print(result.boxes.cls)

print("\n Forma del tensor de máscaras [N_objetos, Alto, Ancho]:")
print(result.masks.data.shape)

# Get detected class names and their Spanish equivalents
class_ids = result.boxes.cls.unique().int().tolist()
class_names_english = [model.names[int(cls_id)] for cls_id in class_ids]

spanish_class_names = {
    'person': 'persona', 'bicycle': 'bicicleta', 'car': 'coche', 'motorcycle': 'motocicleta',
    'airplane': 'avión', 'bus': 'autobús', 'train': 'tren', 'truck': 'camión', 'boat': 'barco',
    'traffic light': 'semáforo', 'fire hydrant': 'boca de incendios', 'stop sign': 'señal de stop',
    'parking meter': 'parquímetro', 'bench': 'banco', 'bird': 'pájaro', 'cat': 'gato',
    'dog': 'perro', 'horse': 'caballo', 'sheep': 'oveja', 'cow': 'vaca', 'elephant': 'elefante',
    'bear': 'oso', 'zebra': 'cebra', 'giraffe': 'jirafa', 'backpack': 'mochila', 'umbrella': 'paraguas',
    'handbag': 'bolso', 'tie': 'corbata', 'suitcase': 'maleta', 'frisbee': 'frisbee', 'skis': 'esquís',
    'snowboard': 'tabla de snowboard', 'sports ball': 'balón deportivo', 'kite': 'cometa',
    'baseball bat': 'bate de béisbol', 'baseball glove': 'guante de béisbol', 'skateboard': 'patineta',
    'surfboard': 'tabla de surf', 'tennis racket': 'raqueta de tenis', 'bottle': 'botella',
    'wine glass': 'copa de vino', 'cup': 'taza', 'fork': 'tenedor', 'knife': 'cuchillo', 'spoon': 'cuchara',
    'bowl': 'cuenco', 'banana': 'plátano', 'apple': 'manzana', 'sandwich': 'sándwich', 'orange': 'naranja',
    'broccoli': 'brócoli', 'carrot': 'zanahoria', 'hot dog': 'perrito caliente', 'pizza': 'pizza',
    'donut': 'rosquilla', 'cake': 'pastel', 'chair': 'silla', 'couch': 'sofá', 'potted plant': 'planta en maceta',
    'bed': 'cama', 'dining table': 'mesa de comedor', 'toilet': 'inodoro', 'tv': 'televisión',
    'laptop': 'portátil', 'mouse': 'ratón', 'remote': 'mando a distancia', 'keyboard': 'teclado',
    'cell phone': 'teléfono móvil', 'microwave': 'microondas', 'oven': 'horno', 'toaster': 'tostadora',
    'sink': 'fregadero', 'refrigerator': 'nevera', 'book': 'libro', 'clock': 'reloj', 'vase': 'jarrón',
    'scissors': 'tijeras', 'teddy bear': 'oso de peluche', 'hair drier': 'secador de pelo', 'toothbrush': 'cepillo de dientes'
}

print("\n🌍 Nombres de las clases detectadas (Inglés y Español):")
for class_name_en in class_names_english:
    class_name_es = spanish_class_names.get(class_name_en, class_name_en) # Default to English if no translation found
    print(f"  - {class_name_en} ({class_name_es})")

### 2.4 Visualización con Supervision

`supervision` hace muy simple la visualización:
- `from_ultralytics(result)` convierte los resultados de YOLO al formato de supervision
- `MaskAnnotator` dibuja la máscara semitransparente
- `PolygonAnnotator` dibuja el contorno del objeto
- `LabelAnnotator` agrega el nombre de la clase en el centro de masa del objeto

In [ ]:
import supervision as sv
from PIL import Image

# Convertir resultado YOLO al formato estándar de Supervision
detections = sv.Detections.from_ultralytics(result)

def annotate(image: Image.Image, detections: sv.Detections) -> Image.Image:
    """
    Anota una imagen con:
    - Máscara de color semitransparente sobre cada objeto detectado
    - Contorno blanco siguiendo el polígono exacto del objeto
    - Etiqueta con el nombre de la clase en el centro de masa
    """
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    mask_annotator    = sv.MaskAnnotator()          # Relleno semitransparente
    polygon_annotator = sv.PolygonAnnotator(color=sv.Color.WHITE)  # Contorno
    label_annotator   = sv.LabelAnnotator(
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        text_position=sv.Position.CENTER_OF_MASS   # Etiqueta en el centro
    )

    out = image.copy()
    out = mask_annotator.annotate(out, detections)
    out = polygon_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections)
    out.thumbnail((1000, 1000))
    return out

annotated_image = annotate(image, detections)
annotated_image

In [ ]:
import numpy as np

def extract_masked_object_with_transparent_background(image: Image.Image, detections: sv.Detections, class_name: str) -> Image.Image:
    """
    Extrae un objeto específico de la imagen usando su máscara de segmentación
    y lo coloca sobre un fondo transparente.
    """
    # 1. Encontrar el índice de la clase solicitada
    target_class_id = None
    for i, cls_id in enumerate(detections.class_id):
        if model.names[cls_id] == class_name:
            target_class_id = i
            break

    if target_class_id is None:
        print(f"No se encontró la clase '{class_name}' en las detecciones.")
        return None

    # 2. Obtener la máscara para el objeto específico
    mask_data = detections.mask[target_class_id]

    # Convertir la imagen a RGBA para soporte de transparencia
    image_rgba = image.convert("RGBA")
    img_array = np.array(image_rgba)

    # Crear una imagen de fondo transparente del mismo tamaño
    transparent_background = np.zeros_like(img_array)

    # Aplicar la máscara: los píxeles dentro de la máscara mantienen su color original,
    # los píxeles fuera de la máscara se vuelven completamente transparentes (alpha=0)
    # La máscara es booleana (True/False), expandirla para que coincida con los canales RGBA
    mask_expanded = mask_data[:, :, np.newaxis]
    img_array_masked = np.where(mask_expanded, img_array, transparent_background)

    return Image.fromarray(img_array_masked, 'RGBA')

# Obtener la imagen del perro con fondo transparente
dog_transparent_image = extract_masked_object_with_transparent_background(image, detections, 'dog')

if dog_transparent_image:
    print(f" Imagen de '{'dog'}' extraída con fondo transparente:")
    display(dog_transparent_image)
else:
    print(f"No se pudo extraer la imagen del '{'dog'}' con fondo transparente.")

---

## 📦 3. Descargar Dataset Personalizado desde Roboflow

### 3.1 Configurar API Key

Para descargar tu dataset necesitas una API Key de Roboflow:

1. Ve a [Roboflow Settings → API](https://app.roboflow.com/settings/api)
2. Copia tu **API Key**
3. En Google Colab: panel izquierdo → ícono 🔑 **Secrets** → Agrega con el nombre `ROBOFLOW_API_KEY`

> 💡 **¿Por qué usamos Roboflow para preparar datos?**  
> Roboflow automatiza todo el pipeline de datos: anotación con IA, control de versiones, aumentaciones automáticas y exportación en el formato exacto de YOLO11. Esto puede ahorrar días de trabajo manual.

### 3.2 Estructura del dataset YOLO para segmentación

```
dataset/
├── data.yaml          ← Rutas de los datos + nombres de las clases
├── train/
│   ├── images/        ← Imágenes de entrenamiento (.jpg, .png)
│   └── labels/        ← Etiquetas de polígonos (.txt)
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

**Formato de las etiquetas de segmentación (`.txt`):**  
Cada línea = un objeto: `clase x1 y1 x2 y2 ... xN yN`  
Las coordenadas son normalizadas (0 a 1) y representan los vértices del polígono.

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

from google.colab import userdata
from roboflow import Roboflow

# PERSONALIZA AQUÍ: Cambia workspace, project y version por los de tu proyecto
# Encuentra el código exacto en Roboflow: tu proyecto → Export → YOLOv11 → Show download code
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("danocarro")
version = project.version(1)
# IMPORTANTE: Usa siempre 'yolov11' como formato de exportación
dataset = version.download("yolov11")

print(f"\n✅ Dataset descargado en: {dataset.location}")

In [ ]:
# Explorar la estructura y contenido del dataset
import os

print(" Archivo data.yaml (configuración del dataset):")
!cat {dataset.location}/data.yaml

print("\n Cantidad de imágenes por split:")
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset.location, split, 'images')
    if os.path.exists(split_path):
        n = len(os.listdir(split_path))
        print(f"  {split:6s}: {n} imágenes")

print("\n Ejemplo de anotación de segmentación (polígono normalizado):")
label_path = os.path.join(dataset.location, 'train', 'labels')
first_label = os.listdir(label_path)[0]
!head -3 {label_path}/{first_label}

---

## 🏋️ 4. Entrenamiento con Early Stopping

### ¿Qué es el Early Stopping y por qué es importante?

Durante el entrenamiento, el modelo mejora progresivamente en cada época. Sin embargo, en algún punto el modelo empieza a **sobreajustarse (overfitting)**: memoriza el dataset de entrenamiento y pierde capacidad de generalizar a nuevas imágenes reales.

El **Early Stopping** monitorea la métrica de validación (mAP de máscaras) y **detiene el entrenamiento automáticamente** si no hay mejora durante `patience` épocas seguidas:

```
Época 10: mAP = 0.72  ← Mejor hasta ahora
Época 11: mAP = 0.71
Época 12: mAP = 0.71
Época 13: mAP = 0.70
Época 14: mAP = 0.70
Época 15: mAP = 0.69  ← 5 épocas sin mejorar → PARA
```

**Ventajas:**
- ⏱️ Reduce el tiempo de entrenamiento (no entrena épocas innecesarias)
- 🧠 Previene el sobreajuste automáticamente
- 💾 Siempre guarda el mejor modelo (`best.pt`) aunque el entrenamiento se haya detenido antes

### 4.1 Implementar el Callback de Early Stopping

YOLO11 ya tiene `patience` integrado como parámetro de entrenamiento. Además, implementamos un **callback personalizado** para tener un log detallado del proceso:

In [ ]:
import numpy as np
from ultralytics import YOLO

class EarlyStoppingCallback:
    """
    Callback de Early Stopping personalizado para YOLO11.

    Monitorea 'metrics/mAP50-95(M)' (mAP de máscaras de segmentación)
    y detiene el entrenamiento si no mejora durante 'patience' épocas.

    Cómo funciona un callback en Ultralytics:
    - Se registra con model.add_callback('evento', funcion)
    - Ultralytics lo llama automáticamente al final de cada época
    - Recibe el objeto 'trainer' con acceso a métricas, épocas, etc.
    """

    def __init__(self, patience: int = 10, min_delta: float = 0.001):
        """
        Args:
            patience  : Épocas a esperar sin mejora antes de detener el entrenamiento
            min_delta : Mejora mínima para considerarse progreso real
        """
        self.patience = patience
        self.min_delta = min_delta
        self.best_metric = -np.inf
        self.epochs_without_improvement = 0
        self.stopped_epoch = None
        self.history = []

    def __call__(self, trainer):
        """
        Función ejecutada al final de cada época de validación.
        Ultralytics llama esta función automáticamente.
        """
        # Leer la métrica de segmentación de máscaras del epoch actual
        current_metric = trainer.metrics.get("metrics/mAP50-95(M)", None)

        if current_metric is None:
            return  # La métrica aún no está disponible en las primeras épocas

        self.history.append(current_metric)
        epoch = trainer.epoch + 1

        if current_metric > self.best_metric + self.min_delta:
            # Hubo mejora significativa — reiniciar contador
            self.best_metric = current_metric
            self.epochs_without_improvement = 0
            print(f"  ✅ Época {epoch:3d}: mAP50-95(M) = {current_metric:.4f}  ← Nuevo mejor")
        else:
            # Sin mejora — incrementar contador
            self.epochs_without_improvement += 1
            remaining = self.patience - self.epochs_without_improvement
            print(f"  ⏳ Época {epoch:3d}: mAP50-95(M) = {current_metric:.4f} "
                  f"| Sin mejora: {self.epochs_without_improvement}/{self.patience} "
                  f"(faltan {remaining} épocas para parar)")

            if self.epochs_without_improvement >= self.patience:
                self.stopped_epoch = epoch
                # Señalizar a Ultralytics que debe detenerse
                trainer.stopper.possible_stop = True
                print(f"\n🛑 EARLY STOPPING activado en época {epoch}")
                print(f"   Mejor mAP50-95(M) obtenido: {self.best_metric:.4f}")
                print(f"   El modelo guardado en best.pt corresponde a la mejor época")

print("✅ Clase EarlyStoppingCallback definida correctamente")

### 4.2 Iniciar el Entrenamiento

**Parámetros clave de entrenamiento:**

| Parámetro | Valor | Descripción |
|---|---|---|
| `model` | `yolo11m-seg.pt` | Modelo base. Opciones: n (nano), s (small), **m (medium)**, l (large), x (xlarge) |
| `data` | `data.yaml` | Archivo de configuración con rutas y nombres de clases |
| `epochs` | `50` | Máximo de épocas (Early Stopping puede parar antes) |
| `imgsz` | `640` | Tamaño de imagen cuadrada en píxeles para entrenamiento |
| `patience` | `10` | Early Stopping integrado de Ultralytics (complementa nuestro callback) |
| `batch` | `16` | Imágenes por lote. Reducir a 8 si hay error de memoria GPU |
| `plots` | `True` | Generar gráficas automáticas de curvas de entrenamiento |

> 💡 **Tranfer Learning:** Al usar `yolo11m-seg.pt` estamos haciendo *fine-tuning* (ajuste fino). El modelo ya sabe detectar objetos genéricos de COCO y ahora aprende a detectar TUS clases específicas. Esto requiere muchos menos datos y tiempo que entrenar desde cero.

In [ ]:
%cd {HOME}

# Instanciar el callback de Early Stopping
early_stopping = EarlyStoppingCallback(patience=10, min_delta=0.001)

# Cargar modelo YOLO11 preentrenado para segmentación
model = YOLO('yolo11m-seg.pt')

# Registrar el callback: se llamará al evento 'on_fit_epoch_end' (fin de cada época)
model.add_callback('on_fit_epoch_end', early_stopping)

print(" Iniciando entrenamiento YOLO11 con Early Stopping...")
print(f"   Modelo     : yolo11m-seg.pt (transfer learning)")
print(f"   Dataset    : {dataset.location}")
print(f"   Épocas máx : 50  |  Patience: 10")
print("-" * 60)

# Lanzar entrenamiento
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    patience=10,        # Early Stopping integrado de Ultralytics
    batch=16,
    plots=True,
    verbose=True,
    project=f'{HOME}/runs/segment',
    name='train'
)

print("\n" + "=" * 60)
print(" ENTRENAMIENTO COMPLETADO")
if early_stopping.stopped_epoch:
    print(f"   Detenido por Early Stopping en época: {early_stopping.stopped_epoch}")
print(f"   Mejor mAP50-95(M): {early_stopping.best_metric:.4f}")
print(f"   Mejor modelo guardado en: {HOME}/runs/segment/train/weights/best.pt")

### 4.3 Visualizar resultados del entrenamiento

YOLO11 genera automáticamente gráficas de entrenamiento en la carpeta `runs/segment/train/`:
- **`results.png`**: Curvas de pérdida (loss) y métricas por época — muestra si el modelo converge correctamente
- **`confusion_matrix.png`**: Qué clases confunde el modelo entre sí — útil para detectar clases problemáticas
- **`val_batch0_pred.jpg`**: Predicciones del modelo sobre el batch 0 del set de validación

In [ ]:
from IPython.display import Image as IPyImage

print(" Archivos generados durante el entrenamiento:")
!ls {HOME}/runs/segment/train/

print("\n Curvas de entrenamiento:")
print("(Loss debería bajar; mAP50, mAP50-95 deberían subir y luego estabilizarse)")
IPyImage(filename=f'{HOME}/runs/segment/train/results.png', width=900)

In [ ]:
print(" Matriz de Confusión Normalizada:")
print("(Diagonal principal alta = buen rendimiento | Off-diagonal = confusiones entre clases)")
IPyImage(filename=f'{HOME}/runs/segment/train/confusion_matrix.png', width=650)

In [ ]:
print(" Predicciones del modelo en el set de validación:")
IPyImage(filename=f'{HOME}/runs/segment/train/val_batch0_pred.jpg', width=850)

---

## ✅ 5. Validación del Modelo Entrenado

La validación evalúa el modelo `best.pt` sobre el set de validación para obtener métricas objetivas y reproducibles:

| Métrica | Descripción |
|---|---|
| **mAP50 (B)** | Precisión media de detección a umbral IoU=0.5 |
| **mAP50-95 (B)** | Precisión media de detección promediada de IoU 0.5 a 0.95 (más exigente) |
| **mAP50 (M)** | Precisión media de **máscara** a umbral IoU=0.5 |
| **mAP50-95 (M)** | Precisión media de **máscara** de IoU 0.5 a 0.95 ← métrica principal de segmentación |

> La letra **(B)** = Bounding Box, **(M)** = Mask. Para segmentación de instancias la métrica relevante es siempre **(M)**.

In [ ]:
print(" Evaluando modelo entrenado sobre el set de validación...")
!yolo task=segment mode=val \
    model={HOME}/runs/segment/train/weights/best.pt \
    data={dataset.location}/data.yaml

---

## 🔮 6. Inferencia con el Modelo Entrenado

### 6.1 Inferencia CLI sobre el set de prueba

In [ ]:
# Ejecutar predicciones sobre todas las imágenes del set de test
!yolo task=segment mode=predict \
    model={HOME}/runs/segment/train/weights/best.pt \
    source={dataset.location}/test/images \
    save=True verbose=False

print("Predicciones guardadas")

### 6.2 Visualización de predicciones en cuadrícula

Mostramos 9 predicciones aleatorias del set de test para inspeccionar visualmente la calidad del modelo. Compara la máscara predicha con lo que esperarías visualmente.

In [ ]:
from ultralytics import YOLO
import supervision as sv
from PIL import Image
import random
import matplotlib.pyplot as plt

# Cargar el mejor modelo entrenado
model = YOLO(f'{HOME}/runs/segment/train/weights/best.pt')

# Cargar el dataset de test con supervision (incluyendo máscaras de polígono)
ds_test = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/test/images",
    annotations_directory_path=f"{dataset.location}/test/labels",
    data_yaml_path=f"{dataset.location}/data.yaml",
    force_masks=True
)

print(f" Dataset de test: {len(ds_test)} imágenes")

In [ ]:
# Paleta de colores para distinguir las clases visualmente
COLORS = sv.ColorPalette.from_hex([
    "#9999ff", "#3399ff", "#66ffff", "#33ff99",
    "#66ff66", "#99ff00", "#ffff00", "#ff9b00",
    "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
])

def annotate(image: Image.Image, detections: sv.Detections) -> Image.Image:
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    mask_annotator    = sv.MaskAnnotator(color=COLORS)
    polygon_annotator = sv.PolygonAnnotator(color=sv.Color.WHITE)
    label_annotator   = sv.LabelAnnotator(
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        text_position=sv.Position.CENTER_OF_MASS
    )
    out = image.copy()
    out = mask_annotator.annotate(out, detections)
    out = polygon_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections)
    out.thumbnail((1000, 1000))
    return out

# Predicciones sobre 9 imágenes aleatorias del test
N = 9
annotated_images = []

for i in random.sample(range(len(ds_test)), min(N, len(ds_test))):
    path, _, _ = ds_test[i]
    image = Image.open(path)
    result = model.predict(image, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    annotated_images.append(annotate(image, detections))

# Mostrar en cuadrícula 3x3
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Predicciones del Modelo YOLO11 (Set de Test)', fontsize=14, fontweight='bold')

for ax, img in zip(axes.flat, annotated_images):
    ax.imshow(img)
    ax.axis("off")

plt.subplots_adjust(wspace=0.02, hspace=0.04)
plt.show()

---

## 🚀 7. Despliegue en Roboflow

Una vez satisfechos con el modelo, lo subimos a **Roboflow Deploy** para:
- Inferencia en la nube sin necesidad de GPU propia
- API REST lista para integrar en apps web, móviles o IoT
- Dashboard de monitoreo de predicciones en producción
- Soporte para dispositivos edge (NVIDIA Jetson, Raspberry Pi)

In [ ]:
print("☁️ Subiendo modelo a Roboflow Deploy...")
project.version(dataset.version).deploy(
    model_type="yolo11",
    model_path=f"{HOME}/runs/segment/train/"
)
print(" Modelo desplegado en Roboflow")

In [ ]:
!pip install -q inference

In [ ]:
from inference import get_model

# Cargar el modelo desde Roboflow Cloud
model_id = project.id.split("/")[1] + "/" + str(dataset.version)
model_cloud = get_model(model_id, ROBOFLOW_API_KEY)

print(f"Modelo cargado desde la nube: {model_id}")

# Inferencia con el modelo en la nube
annotated_cloud = []
for i in random.sample(range(len(ds_test)), min(N, len(ds_test))):
    path, _, _ = ds_test[i]
    image = Image.open(path)
    result = model_cloud.infer(image)[0]
    detections = sv.Detections.from_inference(result)
    annotated_cloud.append(annotate(image, detections))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Predicciones con Modelo en Nube Roboflow', fontsize=14, fontweight='bold')
for ax, img in zip(axes.flat, annotated_cloud):
    ax.imshow(img)
    ax.axis("off")
plt.subplots_adjust(wspace=0.02, hspace=0.04)
plt.show()

---

## 📥 8. Exportar Modelo para Uso Local con OpenCV

Descarga el archivo `best.pt` para usar con la aplicación `video_segmentation.py` que usa OpenCV para inferencia en tiempo real sobre video.

In [ ]:
from google.colab import files
import shutil

best_model = f'{HOME}/runs/segment/train/weights/best.pt'
dest = f'{HOME}/best_yolo11_seg.pt'
shutil.copy(best_model, dest)

print("Descargando modelo entrenado (best_yolo11_seg.pt)...")
files.download(dest)
print("\n Descarga iniciada")
print("💡 Coloca este archivo junto a video_segmentation.py y ejecútalo")

---

## 📋 Resumen del Pipeline Completo

```
📸  1. Recolectar imágenes de tu dominio
         ↓
✏️  2. Anotar con polígonos en Roboflow (Smart Polygon con IA)
         ↓
⚙️  3. Generar versión con aumentaciones en Roboflow
         ↓
📦  4. Exportar en formato YOLOv11 → Descargar en Colab
         ↓
🏋️  5. Entrenar YOLO11 con Early Stopping (para cuando no mejora)
         ↓
📊  6. Validar métricas (mAP50-95 de máscaras)
         ↓
☁️  7. Desplegar en Roboflow o usar localmente
         ↓
📹  8. Aplicación en tiempo real con OpenCV ✅
```

---

*Adaptado por Alfredo Diaz | Basado en [Roboflow Notebooks](https://github.com/roboflow/notebooks) | YOLO11 — Ultralytics*